# Synthetic calibration, QAOA, and the limits of error mitigation

Use a small QAOA problem to separate ideal optimization from device-layout selection, then investigate noise amplification and extrapolation with an analytically checkable circuit. All calibration numbers are synthetic. Learn why gate error estimates are not circuit fidelity and why zero-noise extrapolation trades bias for statistical variance.

**Prerequisites:** Python 3.12 or later and the tested environment in the [collection README](../../../README.md). Install the [root requirements](../../../requirements.txt) into an isolated environment before opening this notebook; select that environment's kernel. Run all cells from a fresh kernel in order. This notebook makes no network calls, reads no external files, and writes no artifacts by default.

**Reproducibility:** all examples and data here are synthetic. Simulator and sampling seeds are explicit. Assertions distinguish mathematical identities, finite-shot checks, and integration contracts. Approximate comparisons state their tolerance.


Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Optimize a three-vertex Max-Cut instance exactly

For a triangle, $C(x)=\sum_{(i,j)}(1-Z_iZ_j)/2$ has maximum cut value 2. A depth-one QAOA state applies $e^{-i\gamma C}$ followed by $e^{-i\beta\sum X}$ to $|+\rangle^{\otimes3}$. Since Qiskit's RZZ$(\theta)=e^{-i\theta ZZ/2}$, the cost layer uses RZZ$(-\gamma)$ up to global phase. We search a finite grid and compare with the uniform baseline, without claiming a global optimizer.


In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error


def show_plot():
    import io

    from IPython.display import Image, display

    buffer = io.BytesIO()
    plt.gcf().savefig(buffer, format="png", dpi=120, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
    plt.close()


edges = [(0, 1), (1, 2), (0, 2)]
cut = np.array([sum(((x >> i) & 1) != ((x >> j) & 1) for i, j in edges) for x in range(8)])
assert cut.max() == 2


def qaoa(gamma, beta):
    qc = QuantumCircuit(3)
    qc.h(range(3))
    for i, j in edges:
        qc.rzz(-gamma, i, j)
    qc.rx(2 * beta, range(3))
    return qc


trials = []
for gamma in np.linspace(0, np.pi, 31):
    for beta in np.linspace(0, np.pi / 2, 31):
        expected = float(Statevector.from_instruction(qaoa(gamma, beta)).probabilities() @ cut)
        trials.append((expected, gamma, beta))
best = max(trials)
assert best[0] > float(cut.mean()) + 0.3 and best[0] <= 2 + 1e-12
print(
    {
        "uniform_expected_cut": float(cut.mean()),
        "grid_best_expected_cut": best[0],
        "gamma": best[1],
        "beta": best[2],
    }
)

## 2. Score layouts against synthetic calibration

For a three-qubit **chain** workload, we search injective placements on a five-qubit directed line. The proxy cost $-\sum_j n_j\log(1-e_j)$ is additive, and its exponential gives an independent-gate survival approximation. This is not a quantum-channel fidelity: it omits coherent errors, idling, readout, and correlated noise. Directed CX calibration can differ by orientation. The QAOA triangle above needs routing on a line; this chain subproblem intentionally isolates layout scoring before adding that extra cost.


In [ ]:
errors = {
    (0, 1): 0.025,
    (1, 0): 0.030,
    (1, 2): 0.040,
    (2, 1): 0.035,
    (2, 3): 0.006,
    (3, 2): 0.008,
    (3, 4): 0.009,
    (4, 3): 0.012,
}
logical = [(0, 1, 3), (1, 2, 2)]


def layout_cost(layout):
    if len(set(layout)) != 3:
        raise ValueError("Layout must be injective")
    if any((layout[a], layout[b]) not in errors for a, b, _ in logical):
        return np.inf
    return sum(-count * np.log1p(-errors[layout[a], layout[b]]) for a, b, count in logical)


scored = [(layout_cost(layout), layout) for layout in itertools.permutations(range(5), 3)]
finite = sorted((cost, layout) for cost, layout in scored if np.isfinite(cost))
best_cost, best_layout = finite[0]
assert best_layout == (2, 3, 4) and best_cost < layout_cost((0, 1, 2))
print(
    pd.DataFrame(
        [
            {"layout": layout, "proxy_cost": cost, "survival_proxy": np.exp(-cost)}
            for cost, layout in finite
        ]
    )
)

## 3. Verify folding before adding noise

For a unitary $U$, the time-ordered sequence $U,U^\dagger,U$ implements U but increases gate exposure. With $U=R_x(\theta)$, odd folding scale $s$ repeats inverse/forward pairs. Optimization must remain at level zero and RX must remain in the simulator basis; otherwise a compiler may cancel the intentionally redundant gates.


In [ ]:
theta = 0.7
scales = np.array([1, 3, 5])


def folded(scale):
    if not isinstance(scale, (int, np.integer)) or scale < 1 or scale % 2 != 1:
        raise ValueError("Use odd positive scale")
    qc = QuantumCircuit(1)
    qc.rx(theta, 0)
    for _ in range((int(scale) - 1) // 2):
        qc.rx(-theta, 0)
        qc.rx(theta, 0)
    return qc


for scale in scales:
    assert np.allclose(Operator(folded(scale)).data, Operator(folded(1)).data, atol=1e-12)
    assert folded(scale).count_ops()["rx"] == scale
p = 0.04
noise = NoiseModel()
noise.add_all_qubit_quantum_error(depolarizing_error(p, 1), ["rx"])
sim = AerSimulator(method="density_matrix", noise_model=noise, basis_gates=["rx"])
circuits = []
for scale in scales:
    qc = folded(scale)
    qc.save_density_matrix()
    circuits.append(qc)
compiled = transpile(circuits, sim, optimization_level=0, seed_transpiler=41)
assert [qc.count_ops().get("rx", 0) for qc in compiled] == list(scales)
result = sim.run(compiled).result()
exact_noisy = np.array(
    [
        np.trace(np.diag([1, -1]) @ np.asarray(result.data(i)["density_matrix"])).real
        for i in range(3)
    ]
)
ideal = float(np.cos(theta))
analytic = ideal * (1 - p) ** scales
assert np.allclose(exact_noisy, analytic, atol=1e-11)
print(pd.DataFrame({"scale": scales, "noisy_Z": exact_noisy, "analytic_Z": analytic}))

## 4. Extrapolation can reduce bias and increase variance

For scales 1,3,5, a quadratic fit's intercept is $(15y_1-10y_3+3y_5)/8$. Negative coefficients amplify sampling noise and may yield an estimate outside the physical interval [-1,1]; clipping hides that fact. Here the exact intercept is closer to the noiseless value for this chosen weak depolarizing model. That observed improvement is not a guarantee for another noise process. Report a standard error alongside the sampled estimate.


In [ ]:
weights = np.array([15, -10, 3]) / 8
assert np.allclose([weights @ np.ones(3), weights @ scales, weights @ (scales**2)], [1, 0, 0])
extrapolated_exact = float(weights @ exact_noisy)
assert abs(extrapolated_exact - ideal) < abs(exact_noisy[0] - ideal)
shots = 50000
rng = np.random.default_rng(41)
sampled = 2 * rng.binomial(shots, (1 + exact_noisy) / 2) / shots - 1
estimate = float(weights @ sampled)
standard_error = float(np.sqrt(np.sum(weights**2 * (1 - exact_noisy**2) / shots)))
assert abs(estimate - extrapolated_exact) < 6 * standard_error
print(
    {
        "ideal": ideal,
        "unmitigated_exact": exact_noisy[0],
        "extrapolated_exact": extrapolated_exact,
        "sampled_ZNE": estimate,
        "sampling_standard_error": standard_error,
    }
)
plt.plot(scales, exact_noisy, "o-", label="Exact noisy values")
plt.errorbar([0], [estimate], yerr=[2 * standard_error], fmt="s", label="ZNE ±2 sampling SE")
plt.axhline(ideal, linestyle="--", color="gray", label="Ideal")
plt.xlabel("Folding scale")
plt.ylabel("⟨Z⟩")
plt.legend()
show_plot()

## 5. Calibrate a readout channel without confusing it with gate noise

Columns of the assignment matrix are prepared states and rows are observed states. For $p_\mathrm{obs}=Mp_\mathrm{true}$, solve a two-by-two linear system using empirical calibration columns. This toy matrix is well conditioned. In higher dimensions, direct inversion can magnify noise or yield negative quasiprobabilities; calibration drift is another source of systematic error.


In [ ]:
M = np.array([[0.94, 0.08], [0.06, 0.92]])
truth = np.array([0.8, 0.2])
observed = M @ truth
calibration_shots = 100000
M_hat = np.column_stack(
    [rng.multinomial(calibration_shots, M[:, j]) / calibration_shots for j in range(2)]
)
observed_hat = rng.multinomial(calibration_shots, observed) / calibration_shots
mitigated = np.linalg.solve(M_hat, observed_hat)
assert np.linalg.cond(M_hat) < 2 and np.isclose(mitigated.sum(), 1)
assert np.max(np.abs(mitigated - truth)) < 0.015
print(
    {
        "condition_number": np.linalg.cond(M_hat),
        "observed": observed_hat,
        "mitigated": mitigated,
        "synthetic_truth": truth,
    }
)

## Limits, cleanup, and next steps

The layout proxy, QAOA grid, depolarizing model, and readout calibration answer different questions. A hardware study must measure calibration drift, routing overhead, idling, correlated noise, shot costs, and extrapolation uncertainty together. The observed synthetic mitigation improvement does not establish hardware performance. No backend calibration, account, remote grader, or cloud submission is used.

No output files or jobs need deletion. Restart the kernel to release results. Extend the study by sweeping p, estimating mean squared error over repeated seeded trials, and comparing unmitigated and extrapolated estimates at equal total shot budgets.

Primary references: [Farhi et al., A Quantum Approximate Optimization Algorithm](https://arxiv.org/abs/1411.4028), [Temme et al., Error Mitigation for Short-Depth Quantum Circuits](https://doi.org/10.1103/PhysRevLett.119.180509), [Qiskit Aer noise models](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html).

**Attribution:** This independent rewrite develops topics from Qiskit Global Summer School 2025. See [course attribution and Apache 2.0 notice](../ATTRIBUTION.md). No course artwork, grading service, saved account, or hardware results are included. This is not an official IBM publication and does not imply IBM endorsement.
